# Tutorial 12: Unified Embedding Outputs

This is the canonical tutorial for `BioEmbedder.embed(...)`. Every example
uses the same public method; only `entity_type`, `model`, and output routing
change.


In [ ]:
from embpy import BioEmbedder

RUN_EMBEDDING = False  # Set True when you are ready to run model inference.
embedder = BioEmbedder(device="auto", organism="human")
print(f"device={embedder.device}")


## 1. One method, several entity types


In [ ]:
genes = ["TP53", "BRCA1", "EGFR"]
smiles = ["CC(=O)Oc1ccccc1C(O)=O", "Cn1c(=O)c2c(ncn2C)n(C)c1=O"]
notes = ["TP53 DNA damage response", "BRCA1 homologous recombination"]

if RUN_EMBEDDING:
    gene_payload = embedder.embed(
        genes, entity_type="gene", id_type="symbol",
        model="esm2_650M", output="payload", key="X_gene_esm2_650M",
    )
    molecule_payload = embedder.embed(
        smiles, entity_type="molecule", id_type="smiles",
        model="morgan_fp", output="payload", key="X_molecule_morgan",
    )
    text_payload = embedder.embed(
        notes, entity_type="text", model="minilm_l6_v2",
        output="payload", key="X_text_minilm",
    )
    print(gene_payload["schema_version"])
    print(molecule_payload["schema_version"])
    print(text_payload["schema_version"])


## 2. AnnData routing rules


In [ ]:
if RUN_EMBEDDING:
    gene_adata = embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model="esm2_650M",
        output="anndata",
        key="X_esm2_650M",
    )
    pert_adata = embedder.embed(
        genes,
        entity_type="perturbation",
        model="subcell_mae_rybg",
        output="anndata",
        attach_to="uns",
        key="X_pert_subcell_mae_rybg",
        morphology_dataset="hpa",
        morphology_source="subcell",
        max_images=3,
    )
    print("gene storage:", gene_adata.varm.keys())
    print("perturbation storage:", pert_adata.uns["perturbations"].keys())


## 3. Cell embeddings use the same method


In [ ]:
import anndata as ad
import numpy as np

cells = ad.AnnData(X=np.abs(np.random.default_rng(0).normal(size=(20, 100))).astype(np.float32))
cells.obs_names = [f"cell_{i}" for i in range(cells.n_obs)]
cells.var_names = [f"Gene_{i}" for i in range(cells.n_vars)]

if RUN_EMBEDDING:
    cells = embedder.embed(
        cells,
        entity_type="cell",
        model="pca",
        output="anndata",
        preprocessing="standard",
        n_pca_components=10,
        n_top_genes=50,
        key="X_pca",
    )
    print(cells.obsm["X_pca"].shape)


## 4. Export choices


In [ ]:
if RUN_EMBEDDING:
    embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model="esm2_650M",
        output="table",
        path="unified_gene_embeddings.zarr",
        fmt="zarr",
    )
